In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("/home/jg2447/slayman/motif_inference/MOBI/scripts/")
from src import plt_func
from src import mobi

### data

In [ ]:
def meme_count(meme_file):

    count = 0
    with open(meme_file, "r") as f:
        for i in f:
            if i.startswith("MOTIF"):
                count += 1
    return count

In [ ]:
# tf names of respective and common tf
fb = [i.replace(".tomtom", "") for i in os.listdir("/home/jg2447/slayman/motif_inference/result/compareMotifDB/factorbook/inference_tomtom") if not i.startswith(".") and i.endswith(".tomtom")]
manolis = [i.replace(".tomtom", "") for i in os.listdir("/home/jg2447/slayman/motif_inference/result/compareMotifDB/manolis/inference_tomtom") if not i.startswith(".") and i.endswith(".tomtom")]
known = [i.replace(".tomtom", "") for i in os.listdir("/home/jg2447/slayman/motif_inference/result/compareMotifDB/baseline/inference_tomtom") if not i.startswith(".") and i.endswith(".tomtom")]
fb_vs_known = set.intersection(set(fb), set(known))
manolis_vs_known = set.intersection(set(manolis), set(known))

In [ ]:
#### factorbook

fb_hits = []
known_hits = []
best_hits = []
for tf in fb_vs_known:
    # for each motif, use the smallest number of predicted motifs among two
    fb_meme_file = "/home/jg2447/slayman/data/motif/factorbook/meme_human_stack/%s.meme" % tf
    fb_count = meme_count(fb_meme_file)
    known_meme_file = "/home/jg2447/slayman/motif_inference/result/compareMotifDB/baseline/inference_motif/%s.meme" % tf
    known_count = meme_count(known_meme_file)
    count = min(known_count, fb_count, 5)
      
    df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/compareMotifDB/factorbook/inference_tomtom/%s.tomtom" % tf, sep="\t")
    df = df[df['p-value'] <= 0.05] # 
    df = df.sort_values('p-value').drop_duplicates("#Query ID") # for one predict motif, only use the best matched known motif
    df = df[df["#Query ID"].str.split("_").str[1].astype(int) < count].copy() # keep top N only, where N is the smallest number of motifs
    fb_hits.append(df.shape[0]) # record number os significant hit motifs
    
    df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/compareMotifDB/baseline/inference_tomtom/%s.tomtom" % tf, sep="\t")
    df = df[df['p-value'] <= 0.05]
    df = df.sort_values('p-value').drop_duplicates("#Query ID")
    motif_file = "/home/jg2447/slayman/motif_inference/result/compareMotifDB/baseline/inference_motif/%s.meme" % tf
    seq2ind = mobi.get_inference_motif_order(motif_file)
    df = df[df["#Query ID"].map(seq2ind) < count].copy()
    known_hits.append(df.shape[0])
    
    df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/compareMotifDB/mobi/inference_tomtom/%s.tomtom" % tf, sep="\t")
    df = df[df['p-value'] <= 0.05]
    df = df.sort_values('p-value').drop_duplicates("#Query ID")
    motif_file = "/home/jg2447/slayman/motif_inference/result/compareMotifDB/mobi/inference_motif/%s.meme" % tf
    seq2ind = mobi.get_inference_motif_order(motif_file)
    df = df[df["#Query ID"].map(seq2ind) < count].copy()
    best_hits.append(df.shape[0])

fb_result = pd.DataFrame([fb_hits, known_hits, best_hits]).T
fb_result.index = fb_vs_known
fb_result.columns = ["Factorbook", "Baseline", "MOBI"]
fb_result.to_csv("./fig6_data_factorbook.txt", sep="\t")

In [ ]:
#### manolis

manolis_hits = []
known_hits = []
best_hits = []
for tf in manolis_vs_known:
    manolis_meme_file = "/home/jg2447/slayman/data/motif/manolis/meme_human_stack/%s.meme" % tf
    manolis_count = meme_count(manolis_meme_file)
    known_meme_file = "/home/jg2447/slayman/motif_inference/result/compareMotifDB/baseline/inference_motif/%s.meme" % tf
    known_count = meme_count(known_meme_file)
    count = min(known_count, manolis_count, 5)
      
    df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/compareMotifDB/manolis/inference_tomtom/%s.tomtom" % tf, sep="\t")
    df = df[df['p-value'] <= 0.05]
    df = df.sort_values('p-value').drop_duplicates("#Query ID")
    df = df[df["#Query ID"].str.split("_").str[1].astype(int) < count].copy()
    manolis_hits.append(df.shape[0])
    
    df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/compareMotifDB/baseline/inference_tomtom/%s.tomtom" % tf, sep="\t")
    df = df[df['p-value'] <= 0.05]
    df = df.sort_values('p-value').drop_duplicates("#Query ID")
    motif_file = "/home/jg2447/slayman/motif_inference/result/compareMotifDB/baseline/inference_motif/%s.meme" % tf
    seq2ind = mobi.get_inference_motif_order(motif_file)
    df = df[df["#Query ID"].map(seq2ind) < count].copy()
    known_hits.append(df.shape[0])
    
    df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/compareMotifDB/mobi/inference_tomtom/%s.tomtom" % tf, sep="\t")
    df = df[df['p-value'] <= 0.05]
    df = df.sort_values('p-value').drop_duplicates("#Query ID")
    motif_file = "/home/jg2447/slayman/motif_inference/result/compareMotifDB/mobi/inference_motif/%s.meme" % tf
    seq2ind = mobi.get_inference_motif_order(motif_file)
    df = df[df["#Query ID"].map(seq2ind) < count].copy()
    best_hits.append(df.shape[0])

manolis_result = pd.DataFrame([manolis_hits, known_hits, best_hits]).T
manolis_result.index = manolis_vs_known
manolis_result.columns = ["Manolis", "Baseline", "MOBI"]
manolis_result.to_csv("./fig6_data_manolis.txt", sep="\t")

### plot

In [ ]:
figsize = (3.2,2)
panel_number_fs = 8
x_tick_label_fs = 5
y_tick_label_fs = 5
x_label_fs = 6
y_label_fs = 6
title_fs = 5
legend_fs = 5

color1 = (0.3,0.9,0.4,0.6)
color2 = (0.3,0.5,0.4,0.6)
color3 = (0.3,0.1,0.4,0.6)

# read data
manolis_result = pd.read_csv("./fig6_data_manolis.txt", sep="\t", index_col=0)
fb_result = pd.read_csv("./fig6_data_factorbook.txt", sep="\t", index_col=0)

sns.set_context("paper")

fig, axs = plt.subplots(1, 2, figsize=figsize, dpi=300, gridspec_kw={'wspace': 0.35})

#### A
ax = axs[0]
ax.bar(
    [0,1,2],
    fb_result.mean().values,
    width=0.7,
    bottom=[0,0,0],
    color=[color1, color2, color3],
    linewidth=0.3,
    edgecolor="black")

_, caps, sticks = ax.errorbar(
    x=[0,1,2],
    y=fb_result.mean().values,
    yerr=fb_result.sem().values,
    fmt=".", lw=1, capsize=1, markersize=0, elinewidth=0.5, ecolor=(0.1, 0.1, 0.1, 0.75))
for cap in caps:
     cap.set_markeredgewidth(0.5)

ax.set_ylim([0,2.85])
ax.set_xlim([-0.7,3])

ax.get_xaxis().set_ticks([0.35,1.35,2.35])
ax.get_xaxis().set_ticklabels(["Factorbook", "Baseline", "MOBI"], rotation=45, ha="right")
ax.get_xaxis().set_tick_params(size=0, pad=2, labelsize=x_tick_label_fs)

ax.get_yaxis().set_ticks([0,0.5,1.0,1.5,2.0,2.5])
ax.get_yaxis().set_tick_params(size=2.5, width=0.5, pad=1.5, labelsize=y_tick_label_fs)
ax.set_ylabel("Correct inference", fontsize=y_label_fs, labelpad=1)

plt_func.ax_trim(ax)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.text(-0.2, 1, "A", transform=ax.transAxes, size=panel_number_fs, weight='bold')
plt_func.barplot_annotate_brackets(ax, 0, 1, "p<0.00035", [0,1,2], fb_result.mean().values+0.2, barh=0.02, dh=0.03)

#### B
ax = axs[1]
ax.bar(
    [0,1,2],
    manolis_result.mean().values,
    width=0.7,
    bottom=[0,0,0],
    color=[color1, color2, color3],
    linewidth=0.3,
    edgecolor="black")

_, caps, sticks = ax.errorbar(
    x=[0,1,2],
    y=manolis_result.mean().values,
    yerr=manolis_result.sem().values,
    fmt=".", lw=1, capsize=1, markersize=0, elinewidth=0.5, ecolor=(0.1, 0.1, 0.1, 0.75))
for cap in caps:
     cap.set_markeredgewidth(0.5)

ax.set_ylim([0,2.85])
ax.set_xlim([-0.7,3])

ax.get_xaxis().set_ticks([0.35,1.35,2.35])
ax.get_xaxis().set_ticklabels(["Kheradpour", "Baseline", "MOBI"], rotation=45, ha="right")
ax.get_xaxis().set_tick_params(size=0, pad=2, labelsize=x_tick_label_fs)

ax.get_yaxis().set_ticks([0,0.5,1.0,1.5,2.0,2.5])
ax.get_yaxis().set_tick_params(size=2.5, width=0.5, pad=1.5, labelsize=y_tick_label_fs)
ax.set_ylabel("Correct inference", fontsize=y_label_fs, labelpad=1)

plt_func.ax_trim(ax)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.text(-0.2, 1, "B", transform=ax.transAxes, size=panel_number_fs, weight='bold')
plt_func.barplot_annotate_brackets(ax, 0, 1, "p=0.18", [0,1,2], manolis_result.mean().values+0.2, barh=0.02, dh=0.03)

plt.savefig("./fig6.pdf", dpi="figure", bbox_inches="tight")